# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns



In [2]:
import sys
print(sys.executable)

c:\Users\abdou\PycharmProjects\payroll_salaries_anomaly_detection\venv\Scripts\python.exe


In [3]:
df = pd.read_csv('DATASET PAIE CSV(5 columns).csv', encoding='latin1')

In [4]:
df.head()

,CSAL_NOSAL,CODE,INTITULE,VALEUR,DATE
0,27,BRUTPAR2,Brut partiel 2,87105.0,8/31/2023 0:00
1,27,NETPAIE,Net à payer,3540000.0,8/31/2023 0:00
2,27,NETIMPO,Net imposable,2742105.0,8/31/2023 0:00
3,27,COTIPAT,Cotisations patronales,506282.0,8/31/2023 0:00
4,27,COTISAL,Cotisations salariales,212105.0,8/31/2023 0:00


In [5]:
df.info()   

<class 'pandas.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 5 columns):
 #   Column      Non-Null Count    Dtype  
---  ------      --------------    -----  
 0   CSAL_NOSAL  1048575 non-null  int64  
 1   CODE        1048575 non-null  str    
 2   INTITULE    1048575 non-null  str    
 3   VALEUR      754603 non-null   float64
 4   DATE        1048575 non-null  str    
dtypes: float64(1), int64(1), str(3)
memory usage: 81.0 MB


In [6]:
df['DATE'] = pd.to_datetime(df['DATE'])

In [7]:
df.head()

,CSAL_NOSAL,CODE,INTITULE,VALEUR,DATE
0,27,BRUTPAR2,Brut partiel 2,87105.0,2023-08-31
1,27,NETPAIE,Net à payer,3540000.0,2023-08-31
2,27,NETIMPO,Net imposable,2742105.0,2023-08-31
3,27,COTIPAT,Cotisations patronales,506282.0,2023-08-31
4,27,COTISAL,Cotisations salariales,212105.0,2023-08-31


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 5 columns):
 #   Column      Non-Null Count    Dtype         
---  ------      --------------    -----         
 0   CSAL_NOSAL  1048575 non-null  int64         
 1   CODE        1048575 non-null  str           
 2   INTITULE    1048575 non-null  str           
 3   VALEUR      754603 non-null   float64       
 4   DATE        1048575 non-null  datetime64[us]
dtypes: datetime64[us](1), float64(1), int64(1), str(2)
memory usage: 67.0 MB


In [15]:
print(df.shape)

(1048575, 5)


In [9]:
intitule_values = df['INTITULE'].unique()
print(f"Unique values in intitule: {intitule_values}")


Unique values in intitule: <ArrowStringArray>
[            'Brut partiel 2',                'Net à payer',
              'Net imposable',     'Cotisations patronales',
     'Cotisations salariales',                       'Brut',
       'Brut pour 13eme mois',         'Net avant retenues',
 'Brut congés année en cours',       'Assiette de la CSG 1',
 ...
                  'RTS15%_02',                  'RTS15%_03',
                  'RTS15%_04',                      'VF_01',
                      'VF_02',         'Autres Retenues_01',
         'Autres Retenues_02',         'Autres Retenues_03',
         'Autres Retenues_04',         'Autres Retenues_05']
Length: 327, dtype: str


In [23]:
df_pivot=df.pivot_table(
                        index=['CSAL_NOSAL','CODE','DATE'],
                        columns='INTITULE',
                        values='VALEUR',
                        aggfunc=['first']).reset_index()

In [24]:
df_pivot = df.sort_values(by='DATE', ascending=True)

In [25]:
df_pivot.head()

,CSAL_NOSAL,CODE,INTITULE,VALEUR,DATE
0,27,BRUTPAR2,Brut partiel 2,87105.0,2023-08-31
10368,256,S_TOT_9,Reconstitution Net av saisie,240005.0,2023-08-31
10369,256,S_TOT__10,RETENUE MOIS 13,240005.0,2023-08-31
10370,256,S_TOT__13,Tot Cotisation Salariale,605010.0,2023-08-31
10371,256,S_TOT__14,TOT RTS,61084.0,2023-08-31


In [10]:
df["normalized_intitule"] = (
    df["INTITULE"]
    .str.strip()
    .str.lower() )
print("Raw unique:", df["INTITULE"].nunique())
print("Normalized unique:", df["normalized_intitule"].nunique())
variants = (
    df.groupby("normalized_intitule")["INTITULE"]
    .nunique()
    .reset_index(name="variant_count")
)

variants = variants[variants["variant_count"] > 1]
print(variants)

variant_list = (
    df.groupby("normalized_intitule")["INTITULE"]
    .unique()
    .reset_index(name="variants_found")
)

variant_list = variant_list[variant_list["variants_found"].apply(len) > 1]

print(variant_list)

Raw unique: 327
Normalized unique: 324
    normalized_intitule  variant_count
27             bonus 02              2
120      frais medicaux              2
293     salaire de base              2
    normalized_intitule                      variants_found
27             bonus 02                [Bonus 02, BONUS 02]
120      frais medicaux    [Frais Medicaux, Frais medicaux]
293     salaire de base  [Salaire de base, SALAIRE DE BASE]


In [11]:
df_clean = df[
    df["VALEUR"].notnull() &
    (df["VALEUR"] != '') &
    (df["VALEUR"] != '1=Oui)"') &
    (df["VALEUR"] != 0)
]

In [12]:
df_clean['DATE'] = pd.to_datetime(df_clean['DATE'])

In [5]:
intitule_values = df_clean['INTITULE'].unique()
print(f"Unique values in intitule: {intitule_values}")

Unique values in intitule: <StringArray>
[                'Brut partiel 2',                    'Net à payer',
                  'Net imposable',         'Cotisations patronales',
         'Cotisations salariales',                           'Brut',
           'Brut pour 13eme mois',             'Net avant retenues',
     'Brut congés année en cours',           'Assiette de la CSG 1',
            'Coût total salarial',            'Coût total patronal',
     'Sous total pour 13ème mois',                 'Assiette RDS 1',
       'Tot Cotisation Salariale',                        'TOT RTS',
     'Congés acquis dans le mois',                'Salaire horaire',
               'Valeur trentième',   'Plafond théorique du salarié',
                'Salaire de base',            'Salaire de présence',
                     'Coût total',    'Net à payer sur la banque 1',
    'Total des heures rémunérées',                        'Horaire',
                       'Présence',   'Total des heures de prés

In [6]:
%%sql
SELECT Distinct INTITULE, COUNT(*) AS OCCURENCES FROM df_clean
GROUP BY INTITULE
ORDER BY OCCURENCES DESC

,INTITULE,OCCURENCES
0,Net à payer,15626
1,Congés acquis dans le mois,15625
2,Brut,15625
3,Net à payer sur la banque 1,15625
4,Historise BRUTABAT,15625
...,...,...
89,Prime Exceptionnelle,1
90,Nombre de jours Préavis,1
91,Cumul libre 43,1
92,Cumul libre 50,1


In [13]:
df_clean = df_clean.drop(columns=['normalized_intitule'])

In [33]:
df[df["VALEUR"].astype(str).str.strip() == '1=Oui)"']

,CSAL_NOSAL,CODE,INTITULE,VALEUR,DATE


In [34]:
df[df["DATE"]==2026]

,CSAL_NOSAL,CODE,INTITULE,VALEUR,DATE


In [44]:
%%sql
SELECT INTITULE, COUNT(*) AS OCCURENCES
FROM df
WHERE DATE >= '2026-01-01'
AND DATE < '2027-01-01'
AND (
    VALEUR IS NULL
    OR VALEUR = '0'
)
GROUP BY INTITULE
ORDER BY OCCURENCES DESC;

,INTITULE,OCCURENCES
0,Solde Congés,2118
1,BONUS OUI/NON,1062
2,Heures complémentaires 2,1062
3,Rappel des salaires 8,1062
4,Rappel des salaires 25,1062
...,...,...
322,Net avant retenues,633
323,Brut pour 13eme mois,633
324,Sous total pour 13ème mois,633
325,Tot Cotisation Salariale,633


In [40]:
%%sql
select * from df where DATE is null

,CSAL_NOSAL,CODE,INTITULE,VALEUR,DATE


In [52]:
%%sql
Select * from df where intitule like 'Recalcul SJR%' and  valeur is null

,CSAL_NOSAL,CODE,INTITULE,VALEUR,DATE
0,7,MAL_RECALC,Recalcul SJR (0=Non 1=Oui),NaN,2026-02-01
1,7,MAT_RECALC,"Recalcul SJR (0=Non, 1=Oui)",NaN,2026-02-01
2,8,MAL_RECALC,Recalcul SJR (0=Non 1=Oui),NaN,2026-02-01
3,8,MAT_RECALC,"Recalcul SJR (0=Non, 1=Oui)",NaN,2026-02-01
4,9,MAL_RECALC,Recalcul SJR (0=Non 1=Oui),NaN,2026-02-01
...,...,...,...,...,...
2119,1162,MAT_RECALC,"Recalcul SJR (0=Non, 1=Oui)",NaN,2026-02-28
2120,1163,MAL_RECALC,Recalcul SJR (0=Non 1=Oui),NaN,2026-02-28
2121,1163,MAT_RECALC,"Recalcul SJR (0=Non, 1=Oui)",NaN,2026-02-28
2122,1164,MAL_RECALC,Recalcul SJR (0=Non 1=Oui),NaN,2026-02-28


In [58]:
%%sql
Select count(*) from df where valeur = 0

,count_star()
0,34911


In [ ]:
df.he